# Analisi Diacronica del Linguaggio tramite Word Embeddings

## Obiettivo del Progetto

Questo notebook documenta la **pipeline completa di analisi diacronica del linguaggio** utilizzando word embeddings CBOW allenati su Google Books N-grams.

L'analisi si divide in **8 fasi principali**:

1. **Configurazione** - Setup dei parametri globali
2. **Preprocessing** - Pulizia e tokenizzazione dei dati grezzi
3. **Costruzione Vocabolario** - Creazione di un vocabolario comune
4. **Dataset PyTorch** - Preparazione dei dati per il training
5. **Training** - Addestramento di modelli CBOW per ogni decennio
6. **Alignment** - Allineamento degli spazi semantici con Procrustes
7. **Semantic Drift** - Analisi dell'evoluzione semantica delle parole
8. **Visualizzazione** - Rappresentazione PCA e t-SNE

---

## Che cos'è il Semantic Drift?

Il **semantic drift** è il cambiamento del significato di una parola nel tempo. Ad esempio:

- **"computer"** negli anni '30 indicava una persona che eseguiva calcoli manualmente
- **"computer"** negli anni '90 indica un dispositivo elettronico

Analizzando i cambiamenti nei vettori di embedding nel tempo, possiamo quantificare e visualizzare questi spostamenti semantici.

---

## Perché Google Books N-grams?

Il dataset **Google Books N-grams v3** fornisce:
- **100+ milioni di libri** digitalizzati
- **Coverage temporale** dal 1900 al 2019
- **Frequenze accurate** per ogni parola in ogni anno
- **Rappresentatività** del linguaggio letterario contemporaneo

Questo dataset è ideale per studi storici sul linguaggio poiché riflette l'uso reale delle parole nel tempo.


## Setup Iniziale

**Tutti i parametri del progetto sono configurati in `src/config.py`**

Parametri principali:
- **Periodo temporale:** `START_YEAR=1900`, `END_YEAR=2019` (12 decenni)
- **N-gram:** `NGRAM_TYPE=5`
- **Vocabolario:** `VOCAB_SIZE=50001`, `UNK_TOKEN='<UNK>'`
- **Embedding:** `EMBEDDING_DIM=300`
- **Directory:** `DATA_PROCESSED_DIR`, `MODELS_DIR`, `ALIGNMENT_ROOT`, `PLOTS_DIR`

Nel notebook importiamo questo file e lo usiamo per tutto.


In [14]:
import os
import sys
import json
import subprocess
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import glob

# Setup path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'config.py').exists():
    PROJECT_ROOT = Path('/home/ccoppola/projects/diachronic_text_analysis')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importa configurazione
from src import config

print(f"✓ Progetto: {PROJECT_ROOT.name}")
print(f"✓ Config importato da: {config.__file__}")


✓ Progetto: diachronic_text_analysis
✓ Config importato da: /home/ccoppola/projects/diachronic_text_analysis/src/config.py


## Parametri di Configurazione

Visualizziamo i parametri principali dal file config.py:


In [15]:
print("\n" + "="*70)
print("PARAMETRI PRINCIPALI - Diachronic Text Analysis")
print("="*70 + "\n")

print(f"📅 PERIODO:        {config.START_YEAR}-{config.END_YEAR} ({len(config.DECADES)} decenni)")
print(f"📊 N-GRAM:          {config.NGRAM_TYPE}-gramma")
print(f"📝 VOCABOLARIO:     {config.VOCAB_SIZE:,} parole + UNK token")
print(f"🧠 EMBEDDING:       dimensione {config.EMBEDDING_DIM}")
print(f"📁 DATA PROCESSED:  {config.DATA_PROCESSED_DIR}")
print(f"🤖 MODELLI:         {config.MODELS_DIR}")

print("\n" + "="*70)



PARAMETRI PRINCIPALI - Diachronic Text Analysis

📅 PERIODO:        1900-2019 (12 decenni)
📊 N-GRAM:          5-gramma
📝 VOCABOLARIO:     50,001 parole + UNK token
🧠 EMBEDDING:       dimensione 300
📁 DATA PROCESSED:  /home/ccoppola/projects/diachronic_text_analysis/data/processed/5gram-full
🤖 MODELLI:         /home/ccoppola/projects/diachronic_text_analysis/models/5gram-full



In [16]:
# Creiamo le directory (se non esistono)
config.create_directories()
print("✓ Directory structure verificata")


✓ Directory structure verificata


## Struttura Dati Disponibile

I dati della pipeline sono già preprocessati e pronti all'uso:


In [17]:
print("="*70)
print("DATI DISPONIBILI NEL PROGETTO")
print("="*70 + "\n")

# Preprocessed data
processed_dir = Path(config.DATA_PROCESSED_DIR)
if processed_dir.exists():
    decade_files = sorted(glob.glob(str(processed_dir / "[0-9][0-9][0-9]0s.txt")))
    print(f"✓ File preprocessati:  {len(decade_files)} decenni")
    
    # Vocab
    vocab_file = Path(config.VOCAB_FILE)
    if vocab_file.exists():
        with open(vocab_file) as f:
            vocab = json.load(f)
        print(f"✓ Vocabolario:        {len(vocab):,} parole")

# Models
models_dir = Path(config.MODELS_DIR)
if models_dir.exists():
    models = sorted(glob.glob(str(models_dir / "emb_*.pt")))
    print(f"✓ Modelli addestrati: {len(models)} CBOW")

print("\n" + "="*70)


DATI DISPONIBILI NEL PROGETTO

✓ File preprocessati:  12 decenni
✓ Vocabolario:        50,002 parole
✓ Modelli addestrati: 12 CBOW



## SEZIONE 4: Il Dataset di Partenza - Google Books N-grams

### Formato dei Dati Raw

Il dataset **Google Books N-grams v3** è fornito in formato **TSV** (Tab-Separated Values):

```
ngram [TAB] year [TAB] match_count [TAB] volume_count
```

**Significato:**
- `ngram`: sequenza di N parole (es: "the quick brown")
- `year`: anno di pubblicazione (1900-2019)
- `match_count`: occorrenze nel corpus per quell'anno
- `volume_count`: numero di libri diversi

**Esempio di riga raw:**
```
the quick brown	1985	1523	47
computer science	1990	892	34
```

Il file raw è molto grande (centinaia di MB). Nel progetto è disponibile in `data/raw/5gram-full/ngrams.txt`


In [19]:
# Cerchiamo il file raw di n-grammi
raw_input_file = Path("data/raw/5gram_expanded/5gram_filtered_expanded.tsv")

print("="*70)
print("ISPEZIONE DATASET RAW")
print("="*70)
print(f"\nFile atteso: {raw_input_file}\n")

if raw_input_file.exists():
    print(f"✓ File trovato!")
    file_size_mb = raw_input_file.stat().st_size / (1024*1024)
    print(f"  - Dimensione: {file_size_mb:.2f} MB")
    
    # Leggi prime righe
    print(f"\n📋 Prime 10 righe del file raw:\n")
    with open(raw_input_file, 'r', encoding='utf-8') as f:
        df_sample = []
        for i, line in enumerate(f):
            if i >= 10:
                break
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                df_sample.append({
                    'ngram': parts[0],
                    'year': int(parts[1]),
                    'match_count': int(parts[2]),
                    'volume_count': int(parts[3])
                })
    
    df_raw = pd.DataFrame(df_sample)
    print(df_raw.to_string(index=False))
    
    # Statistiche sul file
    print(f"\n📊 Statistiche sul file raw:\n")
    
    # Conta righe totali (potrebbe essere lento per file grandi)
    print("  (Conteggio righe in corso...)")
    total_lines = 0
    with open(raw_input_file, 'r', encoding='utf-8') as f:
        for _ in f:
            total_lines += 1
    
    print(f"  - Righe totali: {total_lines:,}")
    print(f"  - Tipo n-gram: {config.NGRAM_TYPE}-grammi")
    print(f"  - Formato: ngram TAB year TAB match_count TAB volume_count")
    
else:
    print(f"✗ File non trovato: {raw_input_file}")
    print(f"\n💡 Il file verrà generato durante la fase di download dei dati.")
    print(f"   Per ora, mostriamo il formato atteso dei dati:")
    
    # Crea un esempio fittizio
    print(f"\n📋 Esempio di formato raw atteso:\n")
    example_data = [
        {'ngram': 'the quick brown', 'year': 1985, 'match_count': 1523, 'volume_count': 47},
        {'ngram': 'computer science', 'year': 1990, 'match_count': 892, 'volume_count': 34},
        {'ngram': 'artificial intelligence', 'year': 1985, 'match_count': 156, 'volume_count': 8},
        {'ngram': 'the quick brown', 'year': 1995, 'match_count': 2341, 'volume_count': 61},
    ]
    df_example = pd.DataFrame(example_data)
    print(df_example.to_string(index=False))

print("\n" + "="*70)


ISPEZIONE DATASET RAW

File atteso: data/raw/5gram_expanded/5gram_filtered_expanded.tsv

✓ File trovato!
  - Dimensione: 17732.17 MB

📋 Prime 10 righe del file raw:

                            ngram  year  match_count  volume_count
account of principal and interest  1903            2             2
account of principal and interest  1904            3             3
account of principal and interest  1905            4             4
account of principal and interest  1908            2             2
account of principal and interest  1909            1             1
account of principal and interest  1913            3             3
account of principal and interest  1914            5             5
account of principal and interest  1915            5             5
account of principal and interest  1916            2             2
account of principal and interest  1917            4             2

📊 Statistiche sul file raw:

  (Conteggio righe in corso...)
  - Righe totali: 512,773,429
  - T

# Fase di Preprocessing

Il preprocessing trasforma i dati **grezzi** in dati **puliti e pronti** per il training. Le trasformazioni applicate includono: lowercasing, rimozione accenti, tokenizzazione, validazione token (lunghezza 2-40 car, >60% alfabetici), filtraggio URL/hashtag e sostituzione numeri con token `NUM`.

Le funzioni principali sono in `src/data/preprocess.py`: `normalize_text()`, `tokenize_and_clean()`, `is_valid_token()`, `process_ngram_line()`.


In [20]:
from src.data.preprocess import normalize_text, tokenize_and_clean, process_ngram_line
import glob

print("="*70)
print("PREPROCESSING - TEST FUNZIONI E RISULTATI")
print("="*70)

# Test su 5 esempi
print("\n📋 TRASFORMAZIONI SU ESEMPI RAW:\n")
print(f"{'Raw Input':<40} → {'Output Pulito':<30}")
print("-" * 72)

test_lines = [
    "The Quick Brown\t1985\t1523\t47",
    "computer SCIENCE\t1990\t892\t34",
    "naïve café\t1995\t234\t12",
    "123 website @mention\t2005\t10\t2",
    "co-operation\t1975\t445\t20",
]

for line in test_lines:
    result = process_ngram_line(line)
    parts = line.split('\t')
    raw = parts[0]
    
    if result:
        tokens, _, _ = result
        output = ' '.join(tokens)
        status = "✓"
    else:
        output = "[SCARTATO]"
        status = "✗"
    
    print(f"{status} {raw:<38} → {output:<28}")

# Risultati preprocessati
print("\n\n📊 FILE PREPROCESSATI GENERATI:\n")
processed_dir = Path(config.DATA_PROCESSED_DIR)
decade_files = sorted(glob.glob(str(processed_dir / "[0-9][0-9][0-9]0s.txt")))

if decade_files:
    decade_stats = []
    for file_path in decade_files:
        decade_name = Path(file_path).stem
        size_mb = Path(file_path).stat().st_size / (1024*1024)
        with open(file_path, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)
        decade_stats.append({'Decennio': decade_name, 'Righe': f"{line_count:,}", 'MB': f"{size_mb:.1f}"})
    
    df_stats = pd.DataFrame(decade_stats)
    print(df_stats.to_string(index=False))
    
    print(f"\n📝 SAMPLE ({Path(decade_files[0]).stem}):")
    with open(decade_files[0], 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(f"  {line.strip()}")

print("\n" + "="*70)


PREPROCESSING - TEST FUNZIONI E RISULTATI

📋 TRASFORMAZIONI SU ESEMPI RAW:

Raw Input                                → Output Pulito                 
------------------------------------------------------------------------
✗ The Quick Brown                        → [SCARTATO]                  
✗ computer SCIENCE                       → [SCARTATO]                  
✗ naïve café                             → [SCARTATO]                  
✗ 123 website @mention                   → [SCARTATO]                  
✗ co-operation                           → [SCARTATO]                  


📊 FILE PREPROCESSATI GENERATI:

Decennio      Righe     MB
   1900s 16,024,078  416.6
   1910s 16,133,329  420.9
   1920s 14,557,403  381.8
   1930s 13,425,888  355.2
   1940s 13,916,515  370.5
   1950s 18,999,042  510.1
   1960s 26,423,829  714.2
   1970s 30,799,543  844.1
   1980s 34,804,869  960.9
   1990s 41,142,587 1135.1
   2000s 48,837,532 1321.2
   2010s 45,113,108 1186.2

📝 SAMPLE (1900s):
  account of 

In [21]:
print("\n" + "="*70)
print("TEST PREPROCESSING SU SINGOLO FILE")
print("="*70)

from src.data.preprocess import aggregate_by_decade
import tempfile

# Crea file TSV di test (20 righe raw)
test_data = """The Quick Brown\t1985\t10\t2
The Quick Brown\t1990\t15\t3
computer SCIENCE\t1985\t8\t1
computer SCIENCE\t1990\t12\t2
naïve café results\t1985\t5\t1
naïve café results\t1990\t7\t1
The machine learns\t1995\t20\t4
machine learning is\t1995\t18\t3
123 website testing\t2000\t3\t1
website testing code\t2000\t5\t1
The Quick Brown\t1995\t25\t5
computer SCIENCE\t1995\t20\t4
artificial intelligence field\t1980\t2\t1
artificial intelligence field\t1985\t8\t2
neural network model\t1990\t6\t1
neural network model\t1995\t10\t2
co-operation agreement made\t1975\t4\t1
co-operation agreement made\t1980\t6\t1
naïve café results\t1975\t3\t1
The Quick Brown\t1980\t12\t2"""

# Scrivi in file temporaneo
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as tmp_input:
    tmp_input.write(test_data)
    test_input_file = tmp_input.name

test_output_dir = tempfile.mkdtemp()

print(f"\n📝 Input: 20 righe raw")
print(f"📂 Output dir: {test_output_dir}\n")

# Esegui preprocessing
aggregate_by_decade(test_input_file, test_output_dir)

# Mostra risultati
from pathlib import Path
test_files = sorted(Path(test_output_dir).glob("*s.txt"))
print(f"\n📊 RISULTATI:\n")
for output_file in test_files:
    with open(output_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"{output_file.name}: {len(lines)} n-grammi")
    for line in lines[:3]:
        print(f"  {line.strip()}")
    if len(lines) > 3:
        print(f"  ... ({len(lines)-3} più)\n")

# Cleanup
import os
os.remove(test_input_file)

print("="*70)



TEST PREPROCESSING SU SINGOLO FILE

📝 Input: 20 righe raw
📂 Output dir: /tmp/tmpupg2bhc9


NLP PREPROCESSING - Word2Vec/CBOW
N-gram: 5, Years: 1900-2019, Min occ: 5
Parameters: len=[2,40], alpha≥0.6, digit≤3, rep≤4

Reading and preprocessing: /tmp/tmpyhho6msu.txt


Processing: 20it [00:00, 93414.34it/s]


Decades found: []


PREPROCESSING COMPLETED


📊 RISULTATI:



In [23]:
import glob

print("="*70)
print("FILE PREPROCESSATI GENERATI")
print("="*70 + "\n")

processed_dir = Path(config.DATA_PROCESSED_DIR)
decade_files = sorted(glob.glob(str(processed_dir / "[0-9][0-9][0-9]0s.txt")))

if decade_files:
    # Tabella statistiche
    print("📊 STATISTICHE PER DECENNIO:\n")
    decade_stats = []
    for file_path in decade_files:
        decade_name = Path(file_path).stem
        size_mb = Path(file_path).stat().st_size / (1024*1024)
        with open(file_path, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)
        decade_stats.append({'Decennio': decade_name, 'Righe': f"{line_count:,}", 'Size MB': f"{size_mb:.2f}"})
    
    df_stats = pd.DataFrame(decade_stats)
    print(df_stats.to_string(index=False))
    
    # Campione di contenuto
    print(f"\n📝 CAMPIONE: Prime 15 righe di {Path(decade_files[0]).name}:\n")
    with open(decade_files[0], 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 15:
                break
            print(f"  {line.strip()}")
    
    total_lines = sum(1 for line in open(decade_files[0], 'r', encoding='utf-8'))
    print(f"\n  ... ({total_lines - 15} righe restanti)")

print("\n" + "="*70)


FILE PREPROCESSATI GENERATI

📊 STATISTICHE PER DECENNIO:

Decennio      Righe Size MB
   1900s 16,024,078  416.62
   1910s 16,133,329  420.91
   1920s 14,557,403  381.83
   1930s 13,425,888  355.23
   1940s 13,916,515  370.47
   1950s 18,999,042  510.05
   1960s 26,423,829  714.25
   1970s 30,799,543  844.08
   1980s 34,804,869  960.86
   1990s 41,142,587 1135.13
   2000s 48,837,532 1321.16
   2010s 45,113,108 1186.22

📝 CAMPIONE: Prime 15 righe di 1900s.txt:

  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  account of principal and interest
  accounting for rents and prof

# Fase di Costruzione del Vocabolario Comune

Quando addestriamo modelli CBOW per diversi decenni, ogni modello ha uno spazio di embedding indipendente. Per confrontare come una parola ha cambiato significato nel tempo, tutti i modelli devono usare lo **stesso vocabolario** con gli **stessi indici** per ogni parola. Altrimenti, i dati sarebbero incomparabili.

Il vocabolario viene costruito aggregando frequenze da TUTTI i decenni: si contano le occorrenze totali di ogni parola (somma across decenni), si ordinano per frequenza decrescente, si selezionano le top 50,000 parole e si assegnano indici 1-50,000. L'indice 0 è riservato al token `<UNK>` per parole fuori vocabolario.

## Struttura del Vocabolario

File: `vocab.json` - Mapping parola → indice (formato JSON)

```json
{
  "<UNK>": 0,
  "the": 1,
  "and": 2,
  ...
}
```

Caratteristiche:
- Total size: 50,001 entries
- Indice 0: `<UNK>` (token sconosciuto)
- Indici 1-50,000: Top 50,000 parole ordinate per frequenza


In [24]:
print("="*70)
print("VOCABOLARIO - STRUTTURA")
print("="*70)

vocab_file = config.VOCAB_FILE

if Path(vocab_file).exists():
    print(f"\n✓ File: {vocab_file}\n")
    
    with open(vocab_file, 'r', encoding='utf-8') as f:
        vocab = json.load(f)
    
    print(f"📊 Dimensione vocabolario: {len(vocab):,} entries\n")
    
    # Mostra prime 10 parole
    word_idx_pairs = [(w, i) for w, i in vocab.items()]
    word_idx_pairs.sort(key=lambda x: x[1])
    
    print(f"Indice | Parola")
    print("-" * 25)
    for word, idx in word_idx_pairs[:10]:
        label = "<UNK>" if word == config.UNK_TOKEN else word
        print(f"{idx:5d} | {label}")
    
    print(f"\n...")
    
else:
    print(f"✗ File non trovato: {vocab_file}")

print("\n" + "="*70)


VOCABOLARIO - STRUTTURA

✓ File: /home/ccoppola/projects/diachronic_text_analysis/data/processed/5gram-full/vocab.json

📊 Dimensione vocabolario: 50,002 entries

Indice | Parola
-------------------------
    0 | <UNK>
    1 | the
    2 | of
    3 | that
    4 | to
    5 | and
    6 | on
    7 | is
    8 | in
    9 | his

...



In [25]:
print("="*70)
print("COME LANCIARE LA COSTRUZIONE DEL VOCABOLARIO")
print("="*70)

build_vocab_command = """
python src/data/build_vocab.py \\
    --processed-dir data/processed/5gram-full \\
    --vocab-size 50001
"""

print("\n📋 Da terminal (dopo preprocessing):\n")
print(build_vocab_command)

print("\nℹ️  Questo comando aggrega le frequenze da tutti i file preprocessati")
print("    e costruisce il vocab.json con le 50,001 parole più frequenti.")
print("    Non viene eseguito qui perché il vocabolario è già generato.")

print("\n" + "="*70)


COME LANCIARE LA COSTRUZIONE DEL VOCABOLARIO

📋 Da terminal (dopo preprocessing):


python src/data/build_vocab.py \
    --processed-dir data/processed/5gram-full \
    --vocab-size 50001


ℹ️  Questo comando aggrega le frequenze da tutti i file preprocessati
    e costruisce il vocab.json con le 50,001 parole più frequenti.
    Non viene eseguito qui perché il vocabolario è già generato.

